# Economy Sim

A notebook which launches an economy simulator.

In [ ]:
import uuid
from decimal import Decimal
import datetime
from pathlib import Path

from configs.config import GLOBAL_ECONOMY_LOG_DIR
from utils.logger import create_custom_logger, override_print_with_logger

from exceptions.not_enough_money_error import NotEnoughMoneyError

from models.enums import CompetitionModel
from models.config_model import WholesalerConfig, RetailerConfig, ConsumerConfig


### Logging

This cell creates a logger, and attaches it to `print` so that 
I can just use `print(f'whatever')` without needing to 
remember to use custom_logger.info.

In [ ]:
now = datetime.datetime.now()
log_path = Path(GLOBAL_ECONOMY_LOG_DIR) / f"{now.year:04d}" / f"{now.month:02d}" / f"{now.day:02d}"
    
economy_logger = create_custom_logger(log_path, logger_name='economy')
original_print = print
override_print_with_logger(economy_logger)
economy_logger.info('Starting economy simulation at {}'.format(now.strftime("%Y-%m-%d %H:%M:%S")))

print_test_statements = True
if print_test_statements:
    # These are test lines to verify that the logger is working correctly. 
    # Set print_test_statements to False to disable them.
    economy_logger.debug("TEST: This is a DEBUG message (only in notebook)")
    economy_logger.info("TEST: This is an INFO message (in both file and notebook)")
    economy_logger.error("TEST: This is an ERROR message (in both file and notebook)")
    print("TEST: This is a test print statement (should appear in both file and notebook)")

### Actor Objects

This cell contains objects for the different types of actors in the economy.

In [ ]:

class Actor:
    name: str
    id: uuid

    def __init__(self, name: str):
        self.name = name
        self.id = uuid.uuid4()
       
       
class BankAccount:
    balance: Decimal
    owner: str
    owner_type: str

    def __init__(self, initial_balance: Decimal = Decimal(0)):
        self.balance = initial_balance



class TransactionLogEntry:
    timestamp: datetime.datetime
    sender: uuid
    sender_type: str
    receiver: uuid
    receiver_type: str
    for_good_service: str
    amount: Decimal
    sender_balance_after: Decimal
    receiver_balance_after: Decimal

    def __init__(self,
                 sender: uuid,
                 sender_type: str,
                 receiver: uuid,
                 receiver_type: str,
                 for_good_service: str,
                 amount: Decimal,
                 sender_balance_after: Decimal,
                 receiver_balance_after: Decimal):
        self.id = uuid.uuid4()
        self.timestamp = datetime.datetime.now()
        self.sender = sender
        self.sender_type = sender_type
        self.receiver = receiver
        self.receiver_type = receiver_type
        self.for_good_service = for_good_service
        self.amount = amount
        self.sender_balance_after = sender_balance_after
        self.receiver_balance_after = receiver_balance_after

class Bank:
    accounts: list[BankAccount]
    transaction_log: list[TransactionLogEntry]

    def __init__(self):
        self.accounts = []
        self.transaction_log = []

    def create_account(self,
                       owner: Actor,
                       initial_balance: Decimal = Decimal(0)) -> BankAccount:
        """
        Create a new bank account for the given owner UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - BankAccount: The newly created bank account for the owner.
        """
        for account in self.accounts:
            if account.owner == owner.id:
                raise ValueError(f'Account already exists for owner {owner.name} (UUID: {owner.id})')
        account = BankAccount(initial_balance=initial_balance)
        account.owner = owner.id
        account.owner_type = type(owner).__name__
        self.accounts.append(account)
        return account
    

    def get_balance(self, owner: uuid) -> Decimal:
        """
        Get the balance of a bank account by the owner's UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - Decimal: The balance of the bank account associated with the given owner UUID.
        Raises:
            - ValueError: If no account is found for the given owner UUID.
        """
        account = self.get_account_by_owner(owner)
        return account.balance
    
    
    def get_account_by_owner(self, owner: uuid) -> BankAccount:
        """
        Get a bank account by the owner's UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - BankAccount: The bank account associated with the given owner UUID.
        Raises:
            - ValueError: If no account is found for the given owner UUID.
        """
        for account in self.accounts:
            if account.owner == owner:
                return account
        raise ValueError(f'No account found for owner {owner}')
    

    def transfer_money(self,
                       sender: uuid,
                       receiver: uuid,
                       amount: Decimal,
                       for_good_service: str) -> None:
        """
        Transfer money from one account to another.

        Args:
            - sender (uuid): The UUID of the sender's account.
            - receiver (uuid): The UUID of the receiver's account.
            - amount (Decimal): The amount of money to transfer.
            - for_good_service (str): One-word description of the good or service this transaction is for (e.g. "orange", "salary", etc.)

        Returns:
            None

        Raises:
            - ValueError: If either the sender or receiver account is not found.
            - NotEnoughMoneyError: If the sender does not have enough money to transfer.
        """
        sender_account = self.get_account_by_owner(sender)
        receiver_account = self.get_account_by_owner(receiver)
        if sender_account.balance < amount:
            raise NotEnoughMoneyError(f'Sender {sender} does not have enough money to transfer {amount}. Current balance: {sender_account.balance}')
        sender_account.balance -= amount
        receiver_account.balance += amount
        transaction_log_entry = TransactionLogEntry(
            sender=sender,
            sender_type=sender_account.owner_type,
            sender_balance_after=sender_account.balance,
            receiver=receiver,
            receiver_type=receiver_account.owner_type,
            receiver_balance_after=receiver_account.balance,
            for_good_service=for_good_service,
            amount=amount
        )
        self.transaction_log.append(transaction_log_entry)





class Retailer(Actor):
    def __init__(self, name: str, retailer_config: RetailerConfig):
        super().__init__(name)
        self.price = -1
        self.stock = 0
        self.competition_model = retailer_config.competition_model
        self.willing_to_sell = True if self.competition_model == CompetitionModel.BERTRAND else False
        if self.competition_model is CompetitionModel.COURNOT:
            raise NotImplementedError('Cournot competition model not implemented yet')
        

    def calculate_demand(self, wholesaler_stock: int, wholesaler_price: Decimal, my_money: Decimal) -> int:
        """
        Calculates the demand for goods from this Retailer based on the wholesaler's
        stock and price.

        The retailer demands as many goods as they can afford at the wholesaler's price.

        Args:
            - wholesaler_stock (int): The current stock available at the wholesaler.
            - wholesaler_price (Decimal): The current price per unit at the wholesaler.
        Returns:
            - int: The quantity of goods the retailer demands from the wholesaler.
        """

        max_affordable_quantity = int(my_money // wholesaler_price)
        demanded_quantity = min(max_affordable_quantity, wholesaler_stock)

        print(f'{self.name} can afford up to {max_affordable_quantity} units at the wholesaler price of {wholesaler_price}. (has {my_money} money)')

        economy_logger.info(f'{self.name} is requesting {demanded_quantity} goods from the wholesaler. (Max affordable: {max_affordable_quantity}, Wholesaler stock: {wholesaler_stock})')
        return demanded_quantity
    

    def calculate_willing_to_sell(self, wholesaler_price: Decimal):
        """
        Determines if this Retailer is willing to sell goods to consumers based
        on the competition model, and the current wholesale price.

        Args:
            - wholesaler_price (Decimal): The current price per unit at the wholesaler.
        Returns:
            - bool: True if the retailer is willing to sell goods to consumers, False otherwise.
        """
        if self.competition_model == CompetitionModel.BERTRAND:
            # in Bertrand competition, retailers are always willing to sell to consumers
            # so long as they have stock, regardless of the wholesale price
            if self.stock <= 0:
                self.willing_to_sell = False
            else:
                self.willing_to_sell = True
        elif self.competition_model == CompetitionModel.COURNOT:
            raise NotImplementedError('Cournot competition model not implemented yet')
        else:
            raise ValueError(f'Unknown competition model: {self.competition_model}')
        return self.willing_to_sell

    
    def receive_goods(self, quantity):
        economy_logger.info(f'{self.name} received {quantity} units from the wholesaler...')
        self.stock += quantity
        return self.stock
    
    def set_price(self, price):
        economy_logger.info(f'{self.name} is setting price to {price}...')
        self.price = price

    def offer_goods(self):
        economy_logger.info(f'{self.name} is offering goods at price {self.price} each...')

    def process_sale(self):
        """
        Process the sale of a retailer's goods to a consumer. Reduce the retailer's
        stock by 1, and if the stock reaches 0, set the willing_to_sell flag to False.

        Returns:
            - None
        Raises:
            - ValueError: If the retailer cannot process the sale because stock is 0 or less
        """
        economy_logger.info(f'{self.name} is processing a sale...')
        if self.stock <= 0:
            raise ValueError(f'{self.name} cannot process sale because stock is {self.stock}')
        self.stock -= 1
        economy_logger.info(f'{self.name} has {self.stock} units left after the sale.')
        if self.stock <= 0:
            economy_logger.info(f'{self.name} has run out of stock and is no longer willing to sell to consumers.')
            self.willing_to_sell = False




class Wholesaler(Actor):
    price: Decimal
    stock: int
    is_unlimited_wholesaler: bool
    stocks: str

    def __init__(self, name, stocks: str = "Orange"):
        super().__init__(name)
        self.price = 0
        self.stock = 0
        self.is_unlimited_wholesaler = True
        self.stocks = stocks

    def set_price(self, price: Decimal) -> None:
        """
        Set the price for the wholesaler's goods.

        Args:
            - price (Decimal): The price to set for the wholesaler's goods.
        Raises:
            - ValueError: If the price is not a Decimal.
        """
        if not isinstance(price, Decimal):
            raise ValueError(f'Price must be a Decimal, got {type(price)}')
        economy_logger.info(f'{self.name} is setting price to {price}...')
        self.price = price

    def set_stock(self, stock: int):
        """
        Set the stock for the wholesaler's goods.

        Args:
            - stock (int): The stock to set for the wholesaler's goods.
        Raises:
            - ValueError: If the stock is not an integer.
        """
        if not isinstance(stock, int):
            raise ValueError(f'Stock must be an integer, got {type(stock)}')
        economy_logger.info(f'{self.name} is setting stock to {stock}...')
        self.stock = stock


    def get_current_price(self):
        """
        Get the current price of the wholesaler's goods.
        """
        return self.price

    def offer_goods(self):
        economy_logger.info(f'{self.name} is selling up to {self.stock} units at price {self.price} each...')
        return self.stock, self.price

    def process_sale(self):
        """
        Process a sale of exactly one of the wholesaler's goods.
        """
        economy_logger.info(f'{self.name} is processing a sale...')
        if self.is_unlimited_wholesaler:
            economy_logger.info(f'{self.name} is an unlimited wholesaler, so stock remains unchanged.')
        else:
            self.stock -= 1


class Consumer(Actor):
    earns_per_iteration: Decimal
    basket_count: int # the number of goods the consumer is holding
    utility: Decimal # the consumer's utility, which increases with each good they consume
    
    base_utility = Decimal('30.0')
    utility_decay = Decimal('0.8') 

    def __init__(self, name, consumer_config: ConsumerConfig):
        super().__init__(name)
        self.earns_per_iteration = consumer_config.earns_per_iteration
        self.basket_count = 0
        self.utility = Decimal('0.0')


    def marginal_utility(self, units_held: int) -> Decimal:
        """
        Utility gained from consuming the next good.

        This decays at a rate of <see cref="utility_decay"/> for each additional good consumed,
        starting from a base utility of <see cref="base_utility"/> for the first good.

        Args:
            - units_held (int): The number of units currently held by the consumer.
        Returns:
            - Decimal: The marginal utility of consuming the next good, based on the number of units currently held.
        """
        return self.base_utility * (self.utility_decay ** units_held)


    def consume_basket(self):
        economy_logger.info(f'{self.name} is consuming their basket of goods...')
        total = sum(self.marginal_utility(i) for i in range(self.basket_count))
        self.utility += total
        self.basket_count = 0
        economy_logger.info(f'{self.name} has consumed their basket and now has total utility of {self.utility}.')


    def willing_to_buy(self, price_point: Decimal) -> bool:
        """
        Check if the Consumer is willing to buy a good at the given price point
        based on their marginal utility. The consumer is willing to buy if the marginal utility
        of consuming the next good is greater than or equal to the price point.

        Args:
            - price_point (Decimal): The price at which the consumer is considering buying a good.
        Returns:
            - bool: True if the consumer is willing to buy at the given price point, False otherwise.     
        """
        return self.marginal_utility(self.basket_count) >= price_point

## The Economy

This cell contains the Economy class. That's the schema for the program, and contains the different types of Actors in the economy, and handles the way they interact with each other.

In [ ]:
from pandas import DataFrame


class Economy:
    wholesaler: Wholesaler = None
    retailers: list[Retailer] = None
    consumers: list[Consumer] = None
    salary_processor: Actor = None
    bank: Bank

    def __init__(self, 
                 wholesaler_config: WholesalerConfig,
                 retailer_config: RetailerConfig,
                 consumer_config: ConsumerConfig):
        self.bank = Bank()
        self.wholesaler = self.setup_wholesaler(wholesaler_config)
        self.retailers = self.setup_retailers(retailer_config)
        self.consumers = self.setup_consumers(consumer_config)
        self.salary_processor = self.setup_salary_processor()




    def pay_salary(self, consumer: Consumer, employer = None):
        """
        Pay a salary to a consumer from the specified employer. If no employer is specified, 
        the salary is paid by the salary processor.

        Args:
            - consumer (Consumer): The consumer receiving the salary.
            - employer (Actor, optional): The employer paying the salary. Defaults to the salary processor.
        """
        if not isinstance(consumer, Consumer):
            raise ValueError(f'Expected consumer recipient of salary to be an instance of Consumer, got {type(consumer)}')
        if employer is None:
            employer = self.salary_processor

        economy_logger.info(f'Paying salary of {consumer.earns_per_iteration} to {consumer.name} from employer {employer.name}...')
        self.bank.transfer_money(employer.id, consumer.id, consumer.earns_per_iteration, for_good_service='salary')
        

    def process_wholesaler_transaction(self, wholesaler: Wholesaler,retailer: Retailer, quantity: int):
        """
        Process a transaction between a retailer and the wholesaler, including money transfer and inventory updates.

        Args:
            - wholesaler (Wholesaler): The wholesaler involved in the transaction.
            - retailer (Retailer): The retailer involved in the transaction.
            - quantity (int): The quantity of goods being purchased by the retailer from the wholesaler.
        """
        total_cost = wholesaler.get_current_price() * quantity
        print(f'{retailer.name} is buying {quantity} units from {wholesaler.name} for a total cost of {total_cost}...')
        try:
            self.bank.transfer_money(retailer.id, wholesaler.id, total_cost, for_good_service=f'wholesale_{wholesaler.stocks}_x_{quantity}')
            for _ in range(quantity):
                wholesaler.process_sale()
            retailer.receive_goods(quantity)
        except NotEnoughMoneyError as e:
            economy_logger.error(f'Transaction failed due to low funds: {e}')


    def process_retailer_transaction(self, retailer: Retailer, consumer: Consumer, good_or_service: str):
        """
        Process a transaction between a retailer and a consumer, including money transfer and inventory updates.

        Args:
            - retailer (Retailer): The retailer involved in the transaction.
            - consumer (Consumer): The consumer involved in the transaction.
            - good_or_service (str): A one-word description of the good or service being sold (e.g. "orange", "apple", "banana", etc.)
        """
        total_cost = retailer.price
        print(f'{consumer.name} is buying 1 unit from {retailer.name} for a total cost of {total_cost}...')
        try:
            self.bank.transfer_money(consumer.id, retailer.id, total_cost, for_good_service=f'retail_{good_or_service}_x_1')
            retailer.process_sale()
            consumer.basket_count += 1
        except NotEnoughMoneyError as e:
            economy_logger.error(f'Transaction failed due to low funds: {e}')





    def run_loop(self, current_iteration: int):
        economy_logger.info(f'Running economy loop for iteration {current_iteration}...')
        
        # wholesaler sets prices and offers goods

        print(f'--- Wholesaler Loop Iteration {current_iteration} ---')
        self.wholesaler.set_price(Decimal('10.0'))
        self.wholesaler.set_stock(1_000_000)
        wholesaler_stock, wholesaler_price = self.wholesaler.offer_goods()

        print(f'--- Retailer Loop Iteration {current_iteration} ---')

        for retailer in self.retailers:
            print(f'{retailer.name} has {retailer.stock} units in stock before the iteration')
            demanded_quantity = retailer.calculate_demand(wholesaler_stock, wholesaler_price, self.bank.get_balance(retailer.id))
            self.process_wholesaler_transaction(self.wholesaler, retailer, demanded_quantity)

            retailer.set_price(wholesaler_price * Decimal('1.5')) # TODO: impl a pricing strategy
            retailer.calculate_willing_to_sell(wholesaler_price)
            print(f'{retailer.name} has {retailer.stock} units in stock after the iteration setup, and is {"willing" if retailer.willing_to_sell else "not willing"} to sell.')



        print(f'--- Salary Payment Loop Iteration {current_iteration} ---')
        for consumer in self.consumers:
            print(f'{consumer.name} earns {consumer.earns_per_iteration} money at the start of the iteration...')
            self.pay_salary(consumer)

        # --- Consumer Loop ---
        print(f'--- Consumer Loop Iteration {current_iteration} ---')
        self.retailers.sort(key=lambda r: r.price)

        # Round-robin: each consumer buys 1 unit per round, loop until termination
        active_consumers = list(self.consumers)  # consumers still able to buy

        while active_consumers:
            consumers_to_remove = []
            
            # Check global termination: no retailers willing to sell
            willing_retailers = [r for r in self.retailers if r.willing_to_sell]
            if not willing_retailers:
                print('No retailers willing to sell. Ending consumer loop.')
                break
            
            for consumer in active_consumers:
                # Find cheapest willing retailer this consumer can afford
                retailer_to_buy_from = None
                for retailer in willing_retailers:
                    if (self.bank.get_balance(consumer.id) >= retailer.price and consumer.willing_to_buy(retailer.price)):
                        retailer_to_buy_from = retailer
                        break
                
                if retailer_to_buy_from is None:
                    consumers_to_remove.append(consumer)
                    continue
                
                # Buy exactly 1 unit (one turn)
                try:
                    self.process_retailer_transaction(retailer_to_buy_from, consumer, good_or_service="Orange")
                except NotEnoughMoneyError:
                    consumers_to_remove.append(consumer)
                    continue
                
                # Refresh willing retailers after the sale
                willing_retailers = [r for r in self.retailers if r.willing_to_sell]
                if not willing_retailers:
                    break
            
            # Remove consumers who can no longer buy
            for c in consumers_to_remove:
                active_consumers.remove(c)
            
        for consumer in self.consumers:
            consumer.consume_basket() # consume the basket at the end of the iteration
            print(f'{consumer.name} has {self.bank.get_balance(consumer.id)} money, and {consumer.utility} utility after the iteration')

        self.iteration_report()
        transaction_df = self.transaction_report()
        save_df_to_csv = True
        if save_df_to_csv:
            csv_path = log_path / f'transactions_iteration_{current_iteration}.csv'
            transaction_df.to_csv(csv_path, index=False)
            economy_logger.info(f'Transaction report for iteration {current_iteration} saved to {csv_path}')

        economy_logger.info(f'Economy loop for iteration {current_iteration} complete.\n\n')


    def setup_wholesaler(self, wholesaler_config: WholesalerConfig):
        wholesaler = Wholesaler(wholesaler_config.name)
        self.bank.create_account(wholesaler, initial_balance=Decimal())
        return wholesaler

    def setup_consumers(self, consumer_config: ConsumerConfig):
        """
        Setup the consumers in the economy based on the provided configuration.

        Each consumer has an earning rate, defined in ConsumerConfig, which determines
        how much money they earn at the start of each iteration.

        Args:
            - consumer_config (ConsumerConfig): The configuration for setting up consumers
        Returns:
            - list[Consumer]: A list of Consumer instances set up according to the configuration.
        """
        def setup_consumer(name, consumer_config):
            return Consumer(name, consumer_config)
        
        consumers = []
        economy_logger.debug(f'Setting up {consumer_config.num_consumers} consumers...')
        for i in range(consumer_config.num_consumers):
            consumer = setup_consumer(f'Consumer {i+1}', consumer_config)
            consumers.append(consumer)
            self.bank.create_account(consumer, initial_balance=Decimal(consumer_config.earns_per_iteration))
        return consumers


    def setup_retailers(self, retailer_config: RetailerConfig):
        def setup_retailer(name):
            return Retailer(name, retailer_config)
        
        retailers = []
        
        economy_logger.debug(f'Setting up {retailer_config.num_retailers} retailers...')

        for i in range(retailer_config.num_retailers):
            retailer = setup_retailer(f'Retailer {i+1}')
            self.bank.create_account(retailer, initial_balance=Decimal(retailer_config.starting_money))
            retailers.append(retailer)
        return retailers
    
    def setup_salary_processor(self):
        """
        Set up the salary processor, which is an abstract Actor which has loads of cash.
        This is used to pay consumers their earnings at the start of each iteration.

        In a more complex economy simulation, salaries would be paid by the retailers and wholesalers
        (employers) based on the work done by the consumers. But for simplicity in this model, 
        there's a single Actor with "unlimited-ish" money who pays out earnings.

        Returns:
            - Actor: The salary processor Actor instance.
        """
        salary_processor = Actor('Salary Processor')
        self.bank.create_account(salary_processor, initial_balance=Decimal(1_000_000_000_000))
        return salary_processor

    def iteration_report(self):
        print('--- Iteration Report ---')

        print(f'Salary Processor: {self.salary_processor.name}, Money: {self.bank.get_account_by_owner(self.salary_processor.id).balance}')
        print(f'Wholesaler: {self.wholesaler.name}, Money: {self.bank.get_account_by_owner(self.wholesaler.id).balance}, Stock: {self.wholesaler.stock}, Price: {self.wholesaler.price}')
        for retailer in self.retailers:
            print(f'Retailer: {retailer.name}, Money: {self.bank.get_account_by_owner(retailer.id).balance}, Stock: {retailer.stock}, Price: {retailer.price}')
        for consumer in self.consumers:
            print(f'Consumer: {consumer.name}, Money: {self.bank.get_balance(consumer.id)}, Utility: {consumer.utility}, Basket Count: {consumer.basket_count}')
        print('--- End of Report ---')

    def transaction_report(self) -> DataFrame:
        print('--- Transaction Report ---')
        transaction_data = [{
            'timestamp': entry.timestamp,
            'sender': entry.sender,
            'sender_type': entry.sender_type,
            'sender_balance_after': entry.sender_balance_after,
            'receiver': entry.receiver,
            'receiver_type': entry.receiver_type,
            'receiver_balance_after': entry.receiver_balance_after,
            'for_good_service': entry.for_good_service,
            'amount': entry.amount
        } for entry in self.bank.transaction_log]
        df = DataFrame(transaction_data)
        print(df)
        print('--- End of Report ---')
        return df

## Application Loop

This cell runs the application in a loop.

In [ ]:
from IPython.display import clear_output
import ipywidgets as widgets


def main(iterations):
    wholesaler_config = WholesalerConfig()
    
    run_competition_model = CompetitionModel.BERTRAND
    retailer_config = RetailerConfig()
    retailer_config.competition_model = run_competition_model
    retailer_config.num_retailers = 3

    consumer_config = ConsumerConfig()
    consumer_config.num_consumers = 15
    consumer_config.earns_per_iteration = Decimal('150.0')
    
    economy = Economy(wholesaler_config, retailer_config, consumer_config)

    print(f'Starting {run_competition_model.name} economy simulation for {iterations} iterations...')
    for i in range(iterations):
        print(f'Running iteration {i + 1}...')
        economy.run_loop(i + 1)
        # clear_output(wait=True)
    print('Economy simulation complete after {} iterations.'.format(iterations))



main(3)